In [ ]:
%pip install -q -U "crewai[google-genai]"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 238.9/238.9 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 724.4/724.4 kB 47.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.5/252.5 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4

In [ ]:

import os
from google.colab import userdata

from crewai import LLM
from crewai.flow.flow import Flow, start, listen, router, or_
from pydantic import BaseModel, Field

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise ValueError(
        "Please add GEMINI_API_KEY in Google Colab Secrets."
    )

os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

llm = LLM(
    model="gemini/gemini-3.1-flash-lite-preview",
    api_key=GEMINI_API_KEY,
    temperature=0.1
)

print("Gemini configured successfully!")


Gemini configured successfully!


In [ ]:
class ResumeScreeningState(BaseModel):
    resumes: list[dict] = Field(default_factory=list)
    current_resume: dict = Field(default_factory=dict)
    score: int = 0
    explanation: str = ""
    status: str = ""
    results: list[dict] = Field(default_factory=list)

In [ ]:
from google.colab import files
from pypdf import PdfReader
from io import BytesIO

uploaded = files.upload()
resumes = []

for filename, content in uploaded.items():
    if not filename.lower().endswith(".pdf"):
        print(f"Skipping non-PDF file: {filename}")
        continue

    try:
        reader = PdfReader(BytesIO(content))
        resume_text = "\n".join(
            page.extract_text() or ""
            for page in reader.pages
        ).strip()

        if resume_text:
            resumes.append({
                "name": filename,
                "text": resume_text
            })
        else:
            print(f"No extractable text found: {filename}")

    except Exception as e:
        print(f"Could not read {filename}: {e}")

if not resumes:
    raise ValueError("Upload at least one readable PDF resume.")

print(f"Successfully loaded {len(resumes)} PDF resume(s).")

Job description loaded.
Number of sample resumes: 3
- Resume_1
- Resume_2
- Resume_3


In [ ]:

from crewai import Agent, Task, Crew
from crewai.flow.flow import Flow, start, listen, router
import json
import pandas as pd
from datetime import datetime

HIGH_SCORE_THRESHOLD = 7


class ResumeScreeningFlow(Flow[ResumeScreeningState]):

    @start()
    def load_resumes(self):
        print("\nLoading PDF resumes...")

        self.state.resumes = resumes
        self.state.results = []

        if not self.state.resumes:
            raise ValueError("No readable PDF resumes were uploaded.")

        print(f"Loaded {len(self.state.resumes)} resumes.")
        return "resumes_loaded"

    @listen(load_resumes)
    def score_resumes(self, _):
        print("\nScoring resumes...")

        for resume in self.state.resumes:
            prompt = f"""
Evaluate the overall quality of this resume on a scale
from 0 to 10.

Consider:
- Clarity and organization
- Demonstrated skills
- Projects and practical experience
- Education and relevant achievements
- Evidence of technical or professional ability

Do not use or assume any job description.
Use only evidence present in the resume.

RESUME:
{resume['text']}

Return ONLY valid JSON:
{{
  "score": 0,
  "explanation": "Brief evidence-based explanation",
  "strengths": ["strength 1", "strength 2"],
  "areas_to_improve": ["area 1", "area 2"]
}}

The score must be an integer between 0 and 10.
"""

            try:
                raw = str(llm.call(prompt)).strip()
                raw = raw.replace("```json", "").replace("```", "").strip()
                data = json.loads(raw)

                score = int(data["score"])
                if not 0 <= score <= 10:
                    raise ValueError("Score must be between 0 and 10.")

                explanation = str(data["explanation"])
                strengths = data.get("strengths", [])
                areas_to_improve = data.get("areas_to_improve", [])

            except (ValueError, TypeError, KeyError, json.JSONDecodeError) as e:
                # Do not treat a failed LLM response as a genuine score of 0.
                score = None
                explanation = f"Scoring failed; manual review required. Details: {e}"
                strengths = []
                areas_to_improve = []

            result = {
                "name": resume["name"],
                "score": score,
                "explanation": explanation,
                "strengths": strengths,
                "areas_to_improve": areas_to_improve,
                "resume_text": resume["text"],
                "status": "Pending routing"
            }

            self.state.results.append(result)

            if score is None:
                print(f"\n{resume['name']}: scoring failed")
            else:
                print(f"\n{resume['name']}: {score}/10")

            print(explanation)

        return "scoring_completed"

    @router(score_resumes)
    def check_score(self):
        # Process one resume per router execution.
        # Failed scores are sent for manual review, not rejected.
        pending = next(
            (
                result for result in self.state.results
                if result["status"] == "Pending routing"
            ),
            None
        )

        if pending is None:
            return "all_processed"

        self.state.current_resume = pending
        self.state.score = pending["score"]
        self.state.explanation = pending["explanation"]

        if self.state.score is None:
            pending["status"] = "Manual review required"
            return "manual_review"

        if self.state.score >= HIGH_SCORE_THRESHOLD:
            print(f"{pending['name']} routed to summary branch.")
            return "shortlist"

        print(f"{pending['name']} routed to rejection-email branch.")
        return "reject"

    @listen(check_score)
    def process_shortlist(self, route):
        if route != "shortlist":
            return

        result = self.state.current_resume

        summary_agent = Agent(
            role="Resume Summary Assistant",
            goal="Summarize resume strengths and areas for improvement",
            backstory=(
                "You prepare concise, evidence-based resume summaries "
                "to help people review candidate qualifications."
            ),
            llm=llm,
            verbose=False
        )

        summary_task = Task(
            description=f"""
Prepare a concise summary of this resume.

RESUME:
{result['resume_text']}

SCREENING SCORE: {result['score']}/10
INITIAL ASSESSMENT: {result['explanation']}

Include:
1. Key skills demonstrated
2. Projects and experience
3. Strengths supported by resume evidence
4. Gaps, missing details, or areas to improve

Do not invent qualifications or make unsupported claims.
""",
            expected_output=(
                "A concise summary covering skills, projects, strengths, "
                "and areas to improve."
            ),
            agent=summary_agent
        )

        summary_crew = Crew(
            agents=[summary_agent],
            tasks=[summary_task],
            verbose=False
        )

        result["summary"] = str(summary_crew.kickoff())
        result["status"] = "Shortlisted for further review"

        print(f"Summary created for {result['name']}.")

    @listen(check_score)
    def process_reject(self, route):
        if route != "reject":
            return

        result = self.state.current_resume

        prompt = f"""
Draft a polite, professional rejection email following an initial
resume screening.

RESUME NAME: {result['name']}
SCREENING SCORE: {result['score']}/10
INITIAL ASSESSMENT: {result['explanation']}

Requirements:
- Use a respectful and professional tone.
- Do not disclose an internal score.
- Do not make unsupported claims about the candidate.
- Keep the message concise.
- Do not invent a company name, role, or recruiter's name.
- Return the email subject and body.
"""

        result["rejection_email"] = str(llm.call(prompt))
        result["status"] = "Rejection email drafted"

        print(f"Rejection email drafted for {result['name']}.")

    @listen(check_score)
    def process_manual_review(self, route):
        if route != "manual_review":
            return

        result = self.state.current_resume
        result["status"] = "Manual review required"

        print(
            f"Scoring failed for {result['name']}; "
            "the resume needs manual review."
        )

    @listen(process_shortlist)
    def continue_after_shortlist(self, _):
        return "resume_processed"

    @listen(process_reject)
    def continue_after_reject(self, _):
        return "resume_processed"

    @listen(process_manual_review)
    def continue_after_manual_review(self, _):
        return "resume_processed"

    @listen(continue_after_shortlist)
    def save_after_shortlist(self, _):
        self.save_results()

    @listen(continue_after_reject)
    def save_after_reject(self, _):
        self.save_results()

    @listen(continue_after_manual_review)
    def save_after_manual_review(self, _):
        self.save_results()

    def save_results(self):
        """Write all accumulated results to results.csv."""
        rows = []

        for result in self.state.results:
            rows.append({
                "Resume": result["name"],
                "Score": (
                    result["score"]
                    if result["score"] is not None
                    else ""
                ),
                "Decision": result["status"],
                "Explanation": result["explanation"],
                "Strengths": "; ".join(
                    str(x) for x in result.get("strengths", [])
                ),
                "Areas to Improve": "; ".join(
                    str(x) for x in result.get("areas_to_improve", [])
                ),
                "Summary": result.get("summary", ""),
                "Rejection Email": result.get("rejection_email", "")
            })

        pd.DataFrame(rows).to_csv("results.csv", index=False)
        print("\nResults saved to results.csv")


# Run the Flow
flow = ResumeScreeningFlow()
flow.kickoff()

# Display results
rows = []
for result in flow.state.results:
    rows.append({
        "Resume": result["name"],
        "Score": result["score"],
        "Decision": result["status"],
        "Explanation": result["explanation"]
    })

display(pd.DataFrame(rows))

# Download the CSV
from google.colab import files
files.download("results.csv")
